# Hallucination Detection - Data Preparation
This notebook prepares the HaluEval QA dataset for supervised binary classification.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

## 1. Load and Explore the Dataset

In [2]:
df = pd.read_json("qa_data.json", lines=True)

df.head()

FileNotFoundError: File qa_data.json does not exist

In [ ]:
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

In [ ]:
print(df.isnull().sum())

## 2. Train-Test Split and Data Restructuring

In [ ]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

print("Training records:", train_df.shape)
print("Testing records:", test_df.shape)

In [ ]:
def restructure_data(data):
    rows = []

    for _, row in data.iterrows():

        # Faithful answer -> label 0
        rows.append({
            "knowledge": row["knowledge"],
            "question": row["question"],
            "answer": row["right_answer"],
            "label": 0
        })

        # Hallucinated answer -> label 1
        rows.append({
            "knowledge": row["knowledge"],
            "question": row["question"],
            "answer": row["hallucinated_answer"],
            "label": 1
        })

    return pd.DataFrame(rows)

In [ ]:
train_data = restructure_data(train_df)
test_data = restructure_data(test_df)

print("Training instances:", train_data.shape)
print("Testing instances:", test_data.shape)

In [ ]:
print("Training class distribution:")
print(train_data["label"].value_counts())

print("\nTesting class distribution:")
print(test_data["label"].value_counts())

In [ ]:
train_data.head()

## 3. Prepare Model Input

In [ ]:

train_data["text"] = (
    "Knowledge: " + train_data["knowledge"] +
    " Question: " + train_data["question"] +
    " Answer: " + train_data["answer"]
)

test_data["text"] = (
    "Knowledge: " + test_data["knowledge"] +
    " Question: " + test_data["question"] +
    " Answer: " + test_data["answer"]
)

train_data[["text", "label"]].head()

In [ ]:

X_train = train_data["text"]
y_train = train_data["label"]

X_test = test_data["text"]
y_test = test_data["label"]

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

In [ ]:
# Check that training and testing records do not overlap

train_questions = set(train_df["question"])
test_questions = set(test_df["question"])

overlap = train_questions.intersection(test_questions)

print("Number of overlapping questions:", len(overlap))

## 4. Save the Processed Dataset

In [ ]:

train_data.to_csv("halu_train.csv", index=False)
test_data.to_csv("halu_test.csv", index=False)

print("Processed datasets saved successfully.")